# Lab Day 10: Qwen3-VL cho định vị hư hại mặt đường
Notebook gọi trực tiếp package `student/`; chạy từ trên xuống. GPU chạy trên Kaggle T4, artifacts/checkpoint lưu trong `/kaggle/working`. Không đưa token vào notebook.

## 0. Môi trường và mã nguồn

In [ ]:
import os, subprocess, json, pathlib, torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU chưa bật')
ROOT = pathlib.Path('/kaggle/working/labrepo')
if not ROOT.exists(): subprocess.run(['git','clone','https://github.com/lenguyentramanh/K4-Track-4-Day10-2A202602760-LeNguyenTramAnh-VisionLanguageModel.git',str(ROOT)],check=True)
%cd /kaggle/working/labrepo
!git pull --ff-only
!pip -q install -r requirements-colab.txt
!nvidia-smi

## 1. Dữ liệu, metadata và kiểm tra nhãn

In [ ]:
# Dataset road-damage đã được gắn ở Kaggle Notebook > Add Input.
from pathlib import Path
import subprocess, sys
DATA_SOURCE=Path('/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data')
assert (DATA_SOURCE/'images').is_dir() and (DATA_SOURCE/'labels-YOLO').is_dir(), f'Dataset input chưa gắn: {DATA_SOURCE}'
subprocess.run([sys.executable,'scripts/prepare_data.py','--source',str(DATA_SOURCE),'--out','data','--work','work'],check=True)

In [ ]:
from student.data import validate_splits
split_info=validate_splits('data')
print('dataset_version:',split_info['dataset_version'])
print(json.dumps(split_info['counts'],indent=2))

In [ ]:
# Kiểm tra số ảnh, lớp và session khi data/splits đã được tạo.
from pathlib import Path
if Path('data/splits/train.jsonl').exists():
    for split in ('train','val','test'):
        rows=load_jsonl(f'data/splits/{split}.jsonl'); print(split, len(rows))
    !python -m student.runner --help

## 2. Smoke checks tọa độ, prompt, mask và module LoRA

In [ ]:
# Các kiểm tra CPU/reward và pipeline nhẹ.
!python -m pytest -q student_tests tests

## 3. E0 zero-shot: validation trước, test sau quality gate

In [ ]:
# Chạy validation để kiểm tra định dạng và baseline; test chỉ sau khi chốt cấu hình.
!python -m student.runner --exp e0 --mode infer --split val

## 4. E1: QLoRA LLM, 300 optimizer steps

In [ ]:
CHECKPOINT_ROOT='/kaggle/working/checkpoints'
!python -m student.runner --exp e1 --mode train --checkpoint-root $CHECKPOINT_ROOT

In [ ]:
!python -m student.runner --exp e1 --mode infer --split val --checkpoint-root $CHECKPOINT_ROOT

## 5. E2/E3: trainable merger và ViT LoRA

In [ ]:
!python -m student.runner --exp e2 --mode train --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e2 --mode infer --split val --checkpoint-root $CHECKPOINT_ROOT

In [ ]:
!python -m student.runner --exp e3 --mode train --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e3 --mode infer --split val --checkpoint-root $CHECKPOINT_ROOT

## 6. Test evidence sau khi validation chọn setup và pass quality gate

In [ ]:
import json
e0=json.load(open('work/e0_val_score.json')); e1=json.load(open('work/e1_val_score.json'))
assert e1['parse_valid_rate']>=0.95, e1['parse_valid_rate']
assert e1['micro']['f1']>=e0['micro']['f1']+0.10, (e0['micro']['f1'],e1['micro']['f1'])
print('Validation gate passed:',e0['micro']['f1'],'->',e1['micro']['f1'])

In [ ]:
# Gate được runner thực thi tự động ngay trước inference test của E1.
!python -m student.runner --exp e0 --mode infer --split test
!python -m student.runner --exp e1 --mode infer --split test --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e2 --mode infer --split test --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e3 --mode infer --split test --checkpoint-root $CHECKPOINT_ROOT

## 7. E4: so sánh train 10% và 100%

In [ ]:
!python -m student.runner --exp e4_data_10 --mode train --train-fraction 0.1 --base-exp auto --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e4_data_10 --mode infer --split val --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e4_data_10 --mode infer --split test --checkpoint-root $CHECKPOINT_ROOT
!python -m student.runner --exp e4_data_100 --mode infer --split test --base-exp auto --checkpoint-root $CHECKPOINT_ROOT

## 8. E5 bonus: reward tests, GRPO smoke, 60 bước, test và reward hacking

In [ ]:
# Chọn base_exp tốt nhất từ E1–E3 theo validation. Smoke chạy riêng, không ghi đè checkpoint GRPO chính.
from student.runner import _select_best_validation
BASE_EXP=_select_best_validation('auto')
!python -m pytest -q student_tests/test_student_e5.py
!python -m student.e5 --mode train --base-exp $BASE_EXP --smoke --max-steps 1 --no-resume
!python -m student.e5 --mode train --base-exp $BASE_EXP --smoke --max-steps 5 --no-resume
!python -m student.e5 --mode train --base-exp $BASE_EXP --max-steps 60 --checkpoint-root $CHECKPOINT_ROOT
!python -m student.e5 --mode infer --split test --baseline-exp $BASE_EXP --checkpoint-root $CHECKPOINT_ROOT
!python -m student.e5 --mode analyze --baseline-exp $BASE_EXP

## 9. Tổng hợp artifacts và báo cáo

In [ ]:
from pathlib import Path
for p in sorted(Path('report/evidence').glob('*.json')): print(p, p.stat().st_size)
!python scripts/check_submission.py
# Sao chép report/evidence, report/figures và checkpoint cần resume ra Kaggle Output trước khi session kết thúc.